# Chapter 24 — Bridge to Multivariate Analytics

## Final Chapter

This chapter connects the statistical methods developed in this repository with the next stage of study: **multivariate analysis**.

> **What happens when relationships among many variables become the main object of analysis?**

PCA and clustering are introduced only as previews. Their theory and full interpretation belong in the **Multivariate-Analytics** repository.

# Part A — From Statistical Analysis to Multivariate Thinking

## 1. Looking Back at Statistical Analytics

Across Chapters 01–23, we learned to:

- describe distributions,
- reason with probability,
- estimate uncertainty,
- test hypotheses,
- compare groups,
- measure relationships,
- build regression models,
- use resampling methods,
- and conduct an end-to-end statistical analysis.

A simplified learning path is:

$$
\text{Describe}
\rightarrow
\text{Compare}
\rightarrow
\text{Infer}
\rightarrow
\text{Relate}
\rightarrow
\text{Model}
$$

## 2. From Univariate to Multivariate Thinking

### Univariate

$$
X
$$

One variable is the main object of analysis.

### Bivariate

$$
X,Y
$$

We study the relationship between two variables.

### Multivariate

$$
X_1,X_2,\ldots,X_p
$$

Many variables are considered together, and the structure among them can itself become the main object of analysis.

## 3. We Already Used Multiple Variables

Multiple linear regression already used several explanatory variables:

$$
Y=
\beta_0+\beta_1X_1+\beta_2X_2+\cdots+\beta_pX_p+\varepsilon
$$

Logistic regression also used multiple predictors.

So why move to multivariate analysis?

The important distinction is not simply the **number of variables**. Regression usually has a clearly defined outcome variable $Y$. Many multivariate methods instead focus on the structure, similarity, or shared information among several variables or observations.

## 4. Prediction vs Structure

A regression problem often looks like:

```text
X1 ─┐
X2 ─┼──→ Y
X3 ─┤
X4 ─┘
```

The outcome $Y$ organizes the analysis.

Multivariate exploration may instead ask:

> **What structure exists among the variables themselves?**

or:

> **Which observations are similar when many features are considered together?**

# Part B — Why Many Variables Are Difficult

## 5. The Challenge of High-Dimensional Data

As the number of variables increases:

- the number of relationships increases,
- visualization becomes difficult,
- variables may contain redundant information,
- interpretation becomes harder,
- and patterns may appear only through combinations of variables.

A table with many columns can contain useful structure that is difficult to see directly.

## 6. The Curse of Dimensionality

A useful conceptual summary is:

$$
p \uparrow
\quad\Rightarrow\quad
\text{Complexity} \uparrow
$$

More variables do not automatically make an analysis easier or better.

In high-dimensional spaces, intuitive ideas about distance, density, and visualization become increasingly difficult. This chapter only introduces that problem; later study will treat it more carefully.

## 7. Correlation Matrix Revisited

For $p$ variables, a correlation matrix has the form:

$$
R=
\begin{bmatrix}
1&r_{12}&\cdots&r_{1p}\\
r_{21}&1&\cdots&r_{2p}\\
\vdots&\vdots&\ddots&\vdots\\
r_{p1}&r_{p2}&\cdots&1
\end{bmatrix}
$$

With only a few variables, this is manageable.

With dozens or hundreds of variables, the central question becomes:

> **Can we summarize these relationships more efficiently?**

# Part C — Multivariate Dataset

## 8. Introduce the Dataset

We will use a synthetic dataset describing **90 fictional regions**.

The variables are socioeconomic-style indices:

- `income_index`
- `education_index`
- `health_index`
- `employment_index`
- `digital_access_index`

All values are synthetic. The regions do not correspond to real countries, cities, or organizations.

## 9. Load and Inspect the Dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing a data/ directory.")

ROOT = find_project_root(Path.cwd())
DATA_PATH = ROOT / "data" / "24_bridge_to_multivariate_analytics.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 10. Explore Many Variables Together

In [ ]:
feature_cols = [
    "income_index",
    "education_index",
    "health_index",
    "employment_index",
    "digital_access_index",
]

df[feature_cols].hist(bins=12, figsize=(10, 7))
plt.tight_layout()
plt.show()

In [ ]:
pd.plotting.scatter_matrix(
    df[feature_cols],
    figsize=(10, 10),
    diagonal="hist",
    alpha=0.65
)
plt.suptitle("Scatter Matrix of the Five Indices", y=1.02)
plt.show()

In [ ]:
corr = df[feature_cols].corr()
corr.round(3)

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)), feature_cols, rotation=45, ha="right")
ax.set_yticks(range(len(feature_cols)), feature_cols)
ax.set_title("Correlation Matrix")

for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

The scatter matrix and correlation matrix contain useful information, but even with only five variables there are many pairwise relationships to inspect.

This motivates methods that summarize **multivariate structure** rather than examining every pair separately.

# Part D — Scaling and Distance

## 11. Why Scaling Matters

Suppose variables are measured on very different scales:

```text
Variable A       0–10
Variable B       0–100
Variable C       0–10,000
```

A distance calculation on the raw values can be dominated by Variable C simply because its numerical scale is larger.

For methods based on distance or variance, scaling can therefore change the result substantially.

## 12. Standardization

A common transformation is the z-score:

$$
z=
\frac{x-\bar{x}}{s}
$$

After standardization, each feature is centered near zero and has a standard deviation near one.

This does **not** make the variables identical. It changes their numerical scale so that one variable does not dominate merely because of its units.

In [ ]:
X = df[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

pd.DataFrame({
    "mean_after_scaling": scaled_df.mean(),
    "std_after_scaling": scaled_df.std(ddof=0)
}).round(3)

## 13. Similarity and Distance

For two observations $\mathbf{x}$ and $\mathbf{y}$, Euclidean distance is:

$$
d(\mathbf{x},\mathbf{y})
=
\sqrt{
\sum_{j=1}^{p}(x_j-y_j)^2
}
$$

A smaller distance means that the observations are closer according to the variables included in the calculation.

Distance depends on:

- which variables are included,
- how they are scaled,
- and which distance definition is used.

In [ ]:
a = X_scaled[0]
b = X_scaled[1]

distance = np.sqrt(np.sum((a - b) ** 2))
print("Euclidean distance between the first two standardized observations:")
print(round(distance, 3))

# Part E — Dimension Reduction

## 14. What Is Dimension Reduction?

Dimension reduction seeks a smaller representation:

$$
X_1,X_2,\ldots,X_p
\rightarrow
Z_1,Z_2,\ldots,Z_k
$$

where

$$
k<p
$$

Possible goals include:

- visualization,
- information compression,
- reducing redundancy,
- and revealing broad patterns.

## 15. Intuition of Principal Component Analysis

Principal Component Analysis asks:

> **Can several correlated variables be summarized by a smaller number of new variables?**

PCA creates new axes called **principal components**.

The first component captures as much variance as possible. The next component captures as much of the remaining variance as possible while being orthogonal to the previous component.

For this bridge chapter, the important ideas are:

- variance,
- linear combinations,
- new coordinate axes,
- and lower-dimensional representation.

## 16. Principal Components

A principal component is a linear combination of the original variables:

$$
PC_1
=
a_{11}X_1
+
a_{12}X_2
+\cdots+
a_{1p}X_p
$$

The coefficients determine how the original variables contribute to the new component.

The detailed mathematics of eigenvalues, eigenvectors, loadings, and component interpretation will be developed in **Multivariate-Analytics**.

## 17. PCA Preview in Python

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df.head()

## 18. Explained Variance — Preview

In [ ]:
explained = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_ratio": np.cumsum(pca.explained_variance_ratio_)
})
explained.round(3)

`explained_variance_ratio_` tells us how much of the total standardized-data variance is represented by each retained principal component.

A two-component representation is useful only if we remember that dimension reduction may discard information.

## 19. Visualize the PCA Result

In [ ]:
fig, ax = plt.subplots()
ax.scatter(pca_df["PC1"], pca_df["PC2"], alpha=0.7)
ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Two-Dimensional PCA Representation")
plt.show()

The PCA plot is not the original five-dimensional dataset. It is a two-dimensional representation.

Useful questions include:

- How much variance is retained?
- Which original variables contribute to each component?
- What information may have been lost?

Those questions become central in a full PCA analysis.

# Part F — Discovering Groups

## 20. From Variables to Observations

Dimension reduction asks how many variables can be summarized.

Another multivariate question is:

> **Are there groups of similar observations?**

This shifts attention from relationships among variables to similarity among rows of the dataset.

## 21. What Is Clustering?

Clustering attempts to group observations without a predefined outcome label.

This is an **unsupervised** problem.

A useful cluster should not merely exist mathematically. It should also be examined for stability, interpretation, and practical meaning.

## 22. K-Means Preview

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X_scaled)

pca_df["cluster"] = clusters
df["cluster"] = clusters

df[["region_id", "cluster"] + feature_cols].head()

K-Means assigns each observation to one of $k$ clusters by iteratively minimizing within-cluster squared distances.

Here we deliberately choose $k=3$ only for a preview.

A real clustering analysis should ask whether that value of $k$ is justified.

## 23. Visualize Clusters

In [ ]:
fig, ax = plt.subplots()

for cluster_id in sorted(pca_df["cluster"].unique()):
    subset = pca_df[pca_df["cluster"] == cluster_id]
    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Cluster {cluster_id}",
        alpha=0.75
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("K-Means Clusters Shown in PCA Space")
ax.legend()
plt.show()

In [ ]:
df.groupby("cluster")[feature_cols].mean().round(1)

The PCA coordinates were used for **visualization**, while K-Means was fitted to the full standardized five-variable data.

This distinction matters:

- PCA summarizes dimensions.
- K-Means assigns observations to groups.
- A PCA plot can help us inspect the clustering result, but the two methods solve different problems.

## 24. Important Questions About Clustering

Before treating clusters as meaningful, ask:

- Why did we choose three clusters?
- Would another value of $k$ change the result?
- Would a different scaling choice change the result?
- Would another distance or clustering method produce different groups?
- Are the clusters stable?
- Are they meaningful in the real context?

These questions are intentionally left open for **Multivariate-Analytics**.

# Part G — Connecting the Two Worlds

## 25. Supervised vs Unsupervised Analysis

| Method | Target Variable | Main Goal |
|---|---|---|
| Linear Regression | Yes | Explain / predict a quantitative outcome |
| Logistic Regression | Yes | Model a binary outcome |
| PCA | No | Reduce dimensions |
| Clustering | No | Discover groups |

This is a useful first distinction, although real analytical workflows can combine supervised and unsupervised methods.

## 26. Choosing an Analysis Method

```text
What is the question?
        |
        +-- Describe data
        |      +-- Descriptive Statistics
        |
        +-- Compare groups
        |      +-- Statistical Tests
        |
        +-- Model quantitative Y
        |      +-- Linear Regression
        |
        +-- Model binary Y
        |      +-- Logistic Regression
        |
        +-- Summarize many variables
        |      +-- PCA
        |
        +-- Discover similar observations
               +-- Clustering
```

The principle from Chapter 23 remains unchanged:

> **Question first, method second.**

## 27. Statistical Analytics and Multivariate Analytics

```text
Statistical-Analytics
|
+-- Describe
+-- Compare
+-- Estimate
+-- Test
+-- Relate
+-- Model
+-- Interpret uncertainty
        |
        v
Multivariate-Analytics
|
+-- Represent multivariate data
+-- Measure similarity
+-- Reduce dimensions
+-- Discover latent structure
+-- Discover groups
```

The second repository does not replace the statistical foundation. It builds on it.

# Part H — Where to Go Next

## 28. What Comes Next

The **Multivariate-Analytics** repository can develop the ideas previewed here in greater depth:

- Multivariate Data
- Matrix Representation
- Standardization
- Distance and Similarity
- Covariance and Correlation Matrices
- Principal Component Analysis
- Explained Variance
- Loadings and Scores
- K-Means Clustering
- Hierarchical Clustering
- Cluster Evaluation
- Interpretation of Multivariate Structure

Chapter 24 is only the bridge.

## 29. Final Learning Map

```text
Raw Data
   |
   v
Understand Variables
   |
   v
Describe Data
   |
   v
Probability
   |
   v
Sampling
   |
   v
Statistical Inference
   |
   v
Hypothesis Testing
   |
   v
Relationships
   |
   v
Regression Modeling
   |
   v
Robustness and Resampling
   |
   v
Comprehensive Statistical Analysis
   |
   v
Multivariate Thinking
```

## 30. Final Summary

This repository began with individual variables and descriptive statistics, then moved through probability, inference, hypothesis testing, relationships, regression, robust methods, and finally an end-to-end statistical analysis.

The next step is to study structure across many variables and observations.

> **Statistics gives us a language for reasoning with uncertainty. Multivariate analysis extends that language to complex relationships among many variables.**

```text
Statistical-Analytics
        |
        v
   Chapter 24
        |
        v
Multivariate-Analytics
```

**End of Statistical-Analytics.**